# Glass-Box dialogue

An overview of the components of Glass-Box:

- the model (the original decision tree),
- the dialogue manager,
- the natural language understanding and
- the assembled system with mock hardware.

In [1]:
from glassbox.model import GlassBoxModel
from glassbox.data import load_personas

model = GlassBoxModel()
personas = load_personas()
for p in personas.values():
    print(p.id,
          p.name,
          model.predict(p.values),
          '(row {} of the UCI data)'.format(p.source['row'])
            if p.source else '(synthetic)')

00 Stu bad (row 54 of the UCI data)
01 Jonathan bad (synthetic)
02 Lurleen bad (row 783 of the UCI data)
03 Ruth bad (row 858 of the UCI data)
04 Joe bad (row 472 of the UCI data)
05 Kirk good (row 60 of the UCI data)
06 Mona good (synthetic)
07 Selma bad (synthetic)
08 Montgomery good (synthetic)


09 Marge good (synthetic)


## The dialogue manager

Requests are intents with arguments, same as the `cmd:` syntax of the chat (e.g., `cmd: why->features_drop:[A9]`).

In [2]:
from glassbox.dialogue import DialogueManager

dialogue = DialogueManager(model)
for intent, args in [('hello', {}),
                     ('apply', {'persona': '03'}),
                     ('outcome', {}),
                     ('why', {}),
                     ('why', {'features_drop': ['A5']}),
                     ('fairness', {}),
                     ('action', {}),
                     ('myself', {'features': ['A9', 'A13']})]:
    print('>', intent, args)
    print(dialogue.process_intent(intent, args), '\n')

> hello {}
Hello. Welcome to Robo Bank. If you would like to apply for a loan please provide your name or application number. Alternatively please scan your application card. 

> apply {'persona': '03'}
Thank you for your application, Ruth. Please bear with me a moment while I process your application. Your loan application has been processed. Would you like to know the outcome? 

> outcome {}
Your application has been declined. 

> why {}
I do not have any ongoing enquiry. Please give me a moment to open one for you. Your application has been declined. If you had asked for at most 883 or between 955 and 1181 or between 1286 and 1917 or between 2111 and 2578 pounds, rather than 3959 pounds, your loan would be accepted. 

> why {'features_drop': ['A5']}
I do not have any ongoing enquiry. Please give me a moment to open one for you. Your loan would be accepted if you had requested your loan for at most 11 or between 23 and 43 months, rather than 15 months. (When avoiding using: loan amou

"Tell me about myself" without naming a feature depends on the output channel:
all the features in the chat, only the current account status when speaking
(spoken answers are kept short).

In [3]:
print('SAY:')
print(dialogue.process_intent('myself', channel='speech'))
print('\nWRITE:')
print(dialogue.process_intent('myself', channel='chat'))

SAY:
Dear Ruth, at the time of application: you have had overdraft on your current account.

WRITE:
Dear Ruth, at the time of application: you have requested your loan for 15 months and you have asked for 3959 pounds and you have had overdraft on your current account and you have had less than 100 pounds on your savings account and you have not had other loans and you have had existing loans paid back on time until now and you have received an instalment rate of 3 percent and you have been skilled employee and you have been employed between 1 and 4 years and you have been a foreign worker and you have owned a house and you have been 29 years old and you have been female (divorced or separated or married).


## Natural language understanding

The default backend reimplements the original Rasa NLU pipeline with spaCy, a CRF and an SVM.

In [4]:
from glassbox.nlu import Intentioner, get_backend

intentioner = Intentioner(get_backend('spacy'))
for sentence in ['Hello',
                 'My application number is four',
                 'Is the decision fair?',
                 'Why given my age and despite my marital status?',
                 'What can I do to get my loan accepted?']:
    print(repr(sentence), '->', intentioner.sentence_to_intent(sentence))

'Hello' -> ('hello', {})
'My application number is four' -> ('apply', {'persona': '04'})
'Is the decision fair?' -> ('fairness', {})
'Why given my age and despite my marital status?' -> ('why', {'features_require': ['A13'], 'features_drop': ['A9']})
'What can I do to get my loan accepted?' -> ('action', {})


## The assembled system (with mock hardware)

In [5]:
import logging
logging.basicConfig(level=logging.INFO, format='%(name)s: %(message)s', force=True)
from glassbox.config import load_config
from glassbox.core import GlassBox

box = GlassBox(load_config(overrides={
    'hardware': {
        'leds': {
            'backend': 'mock'
        },
        'speech': {
            'backend': 'mock'
        }
    }}))
print(box.handle_message('hello'))
print(box.handle_qr('04'))
print(box.handle_message('what is the outcome'))
print(box.leds.history[-1], box.speaker.said)

glassbox.hardware.leds: LED ring: rainbow {'clockwise': True}


glassbox.hardware.leds: LED square: rainbow 


glassbox.hardware.leds: LED ring: still 


glassbox.hardware.leds: LED square: still 


glassbox.hardware.leds: LED ring: still 


glassbox.hardware.leds: LED square: still 


... (5 outputs left out)

glassbox.hardware.leds: LED ring: still 


glassbox.hardware.leds: LED square: still 


glassbox.hardware.leds: LED square: ascii {'ascii_pic': 'no', 'colour': 16711680, 'scroll': False}


Hello. Welcome to Robo Bank. If you would like to apply for a loan please provide your name or application number. Alternatively please scan your application card.
Thank you for your application, Joe. Please bear with me a moment while I process your application. Your loan application has been processed. Would you like to know the outcome?
Your application has been declined.
{'led_matrix': 'square', 'pattern': 'ascii', 'params': {'ascii_pic': 'no', 'colour': 16711680, 'scroll': False}} ['Thank you for your application, Joe. Please bear with me a moment while I process your application. Your loan application has been processed. Would you like to know the outcome?']
